### How many taxi trips will occur in each zone, each hour?


In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import logging, os, warnings
warnings.filterwarnings("ignore")

In [2]:
builder = (
    SparkSession.builder
        .appName("ForecastDemand")

        # Delta Lake
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

        # Memory (CRITICAL for 100M+ rows)
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")

        # CPU cores
        .config("spark.driver.cores", "4")

        # Shuffle tuning
        .config("spark.sql.shuffle.partitions", "200")

        # Adaptive query optimization
        .config("spark.sql.adaptive.enabled", "true")

        # Prevent huge driver result crashes
        .config("spark.driver.maxResultSize", "2g")

        #UI tuning
        .config("spark.ui.showConsoleProgress", "false")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()


In [3]:
log_dir = "../../logs"
os.makedirs(log_dir, exist_ok=True)
logging.basicConfig(
    filename=os.path.join(log_dir, "forecast_demand_model_training.log"),
    level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s"
)

In [4]:
df = (
    spark.read.format("delta")
    .load(r"..\..\04_storage_platinum\demand_series")
    .toPandas()
)

In [5]:
# Build proper datetime column for sorting
df["ds"] = (
    pd.to_datetime(df["trip_date"]) +
    pd.to_timedelta(df["pickup_hour"], unit="h")
)
df = df.sort_values(["pu_location_id", "ds"]).reset_index(drop=True)

logging.info(f"Loaded demand_series: {len(df)} rows, {df['pu_location_id'].nunique()} zones")


In [6]:
# --------------------------------------------------
# TRAIN / TEST SPLIT
# Last 30 days = test (holdout); everything else = train
# This simulates a real deployment: model trained on
# historical data, evaluated on unseen future dates.
# --------------------------------------------------

CUTOFF     = df["ds"].max() - pd.Timedelta(days=30)
train_df   = df[df["ds"] <= CUTOFF].dropna(subset=["lag_1h", "lag_24h", "lag_168h"])
test_df    = df[df["ds"] >  CUTOFF].dropna(subset=["lag_1h", "lag_24h", "lag_168h"])

logging.info(f"Train: {len(train_df)} rows | Test: {len(test_df)} rows")

In [7]:
# --------------------------------------------------
# FEATURES
# zone_id is passed as categorical — LightGBM learns
# per-zone demand level without one-hot encoding,
# which would explode dimensionality at 265 zones.
# --------------------------------------------------

FEATURE_COLS = [
    "pu_location_id",      # categorical — zone identity
    "pickup_hour",
    "day_of_week",
    "month",
    "trip_year",
    "is_weekend",
    "is_rush_hour",
    "is_night",
    "lag_1h",
    "lag_24h",
    "lag_168h",
    "avg_fare",
    "avg_distance",
    "avg_passengers",
]
TARGET = "trip_count"

X_train = train_df[FEATURE_COLS]
y_train = train_df[TARGET]
X_test  = test_df[FEATURE_COLS]
y_test  = test_df[TARGET]

### LIGHTGBM — GLOBAL MODEL
##### One model, all zones. pu_location_id is categorical.

In [8]:
lgb_train = lgb.Dataset(
    X_train, label=y_train,
    categorical_feature=["pu_location_id"],
    free_raw_data=False
)
lgb_val = lgb.Dataset(
    X_test, label=y_test,
    categorical_feature=["pu_location_id"],
    reference=lgb_train,
    free_raw_data=False
)

In [9]:
params = {
    "objective":        "regression_l1",   # MAE loss — robust to demand spikes
    "metric":           ["mae", "rmse"],
    "learning_rate":    0.05,
    "num_leaves":       127,               # 2^7 — balanced depth for tabular data
    "min_child_samples": 20,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq":     5,
    "reg_alpha":        0.1,
    "reg_lambda":       1.0,
    "verbose":          -1,
    "n_jobs":           -1,
    "seed":             42,
}

In [10]:
callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=100)
]

In [11]:
logging.info("Training global LightGBM model...")
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=["train", "val"],
    callbacks=callbacks
)
logging.info(f"Best iteration: {model.best_iteration}")

Training until validation scores don't improve for 50 rounds
[100]	train's l1: 6.21309	train's rmse: 14.896	val's l1: 8.57394	val's rmse: 19.5614
[200]	train's l1: 5.80164	train's rmse: 13.4502	val's l1: 7.97316	val's rmse: 17.4381
[300]	train's l1: 5.6367	train's rmse: 12.9636	val's l1: 7.65436	val's rmse: 16.3945
[400]	train's l1: 5.56605	train's rmse: 12.7734	val's l1: 7.4961	val's rmse: 15.8979
[500]	train's l1: 5.50798	train's rmse: 12.618	val's l1: 7.39821	val's rmse: 15.5594
[600]	train's l1: 5.45354	train's rmse: 12.481	val's l1: 7.30985	val's rmse: 15.2773
[700]	train's l1: 5.41067	train's rmse: 12.3738	val's l1: 7.24457	val's rmse: 15.0751
[800]	train's l1: 5.37812	train's rmse: 12.2937	val's l1: 7.19521	val's rmse: 14.9331
[900]	train's l1: 5.36616	train's rmse: 12.2673	val's l1: 7.18226	val's rmse: 14.8786
Early stopping, best iteration is:
[890]	train's l1: 5.36704	train's rmse: 12.2681	val's l1: 7.17748	val's rmse: 14.8824


### Evaluation

Global

In [12]:
preds = np.clip(model.predict(X_test), 0, None)

mae  = mean_absolute_error(y_test, preds)
rmse = root_mean_squared_error(y_test, preds)
r2   = r2_score(y_test, preds)
mape = np.mean(
    np.abs((y_test.values - preds) /
           np.where(y_test.values == 0, 1, y_test.values))
) * 100

print("\n" + "="*45)
print("GLOBAL MODEL — TEST SET PERFORMANCE")
print("="*45)
print(f"  MAE  : {mae:.2f} trips/hour")
print(f"  RMSE : {rmse:.2f} trips/hour")
print(f"  R²   : {r2:.4f}")
print(f"  MAPE : {mape:.2f}%")
logging.info(f"Global — MAE={mae:.2f} RMSE={rmse:.2f} R²={r2:.4f} MAPE={mape:.2f}%")


GLOBAL MODEL — TEST SET PERFORMANCE
  MAE  : 7.18 trips/hour
  RMSE : 14.88 trips/hour
  R²   : 0.9630
  MAPE : 33.43%


Demand based MAPE

In [14]:
# Split MAPE by demand level — shows model is accurate where it matters
test_df['prediction'] = preds

test_df["demand_bucket"] = pd.cut(
    test_df["trip_count"],
    bins=[0, 10, 50, 200, float("inf")],
    labels=["very low (<10)", "low (10–50)", "medium (50–200)", "high (200+)"],
    include_lowest=True
)

mape_by_bucket = (
    test_df.groupby("demand_bucket")
    .apply(lambda g: np.mean(
        np.abs((g["trip_count"] - g["prediction"]) /
               np.where(g["trip_count"] == 0, 1, g["trip_count"])) * 100
    ))
    .reset_index(name="mape")
)
print("\nMAPE by Demand Bucket:")
print(mape_by_bucket)


MAPE by Demand Bucket:
     demand_bucket       mape
0   very low (<10)  49.433747
1      low (10–50)  24.308259
2  medium (50–200)  14.583443
3      high (200+)  11.218864


Per - Zone

In [15]:
test_df = test_df.copy()
test_df["prediction"] = preds

per_zone = []
for zone_id, grp in test_df.groupby("pu_location_id"):
    y_z  = grp[TARGET].values
    p_z  = grp["prediction"].values
    per_zone.append({
        "zone_id":  zone_id,
        "mae":      round(mean_absolute_error(y_z, p_z), 2),
        "rmse":     round(root_mean_squared_error(y_z, p_z), 2),
        "r2":       round(r2_score(y_z, p_z), 4),
        "mape":     round(np.mean(np.abs((y_z - p_z) /
                         np.where(y_z == 0, 1, y_z))) * 100, 2)
    })

per_zone_df = pd.DataFrame(per_zone).sort_values("mae")
print("\nPer-zone results (sorted by MAE):")
print(per_zone_df.to_string(index=False))


Per-zone results (sorted by MAE):
 zone_id   mae  rmse     r2  mape
      93  0.14  0.39 0.7961  8.78
      12  0.45  1.03 0.7926 18.15
      10  0.72  1.21 0.6340 31.96
      71  1.05  1.62 0.6373 40.82
     197  1.09  1.66 0.5917 44.53
     216  1.12  1.73 0.5437 42.23
      91  1.16  1.92 0.6431 38.36
      72  1.25  1.90 0.6556 43.02
      35  1.35  2.05 0.6390 41.95
     223  1.35  2.08 0.4799 45.32
     168  1.36  1.99 0.4945 49.87
      95  1.37  2.06 0.4787 49.78
     177  1.40  2.11 0.5184 48.52
     260  1.45  2.27 0.4050 49.39
      82  1.53  2.37 0.4843 50.61
      36  1.59  2.67 0.8000 48.82
      39  1.60  2.44 0.6706 45.78
     264  1.68  2.44 0.7539 33.43
     179  1.76  2.73 0.5666 45.96
      17  1.85  2.61 0.5454 50.99
     146  1.88  3.03 0.4569 46.02
      49  1.95  2.82 0.5659 50.64
     130  1.95  2.80 0.4773 53.46
      65  1.96  2.91 0.6311 39.68
     243  1.99  2.96 0.6083 41.95
     129  2.02  3.00 0.4762 44.83
      66  2.05  3.40 0.7124 38.25
     188  2.0

Feature Importance

In [16]:
importance_df = pd.DataFrame({
    "feature":    model.feature_name(),
    "importance": model.feature_importance(importance_type="gain")
}).sort_values("importance", ascending=False)

print("\nFeature importances (gain):")
print(importance_df.to_string(index=False))
logging.info("Feature importances:\n" + importance_df.to_string())


Feature importances (gain):
       feature   importance
      avg_fare 1.714518e+08
  avg_distance 3.963248e+07
      lag_168h 3.486295e+07
        lag_1h 2.782890e+07
avg_passengers 1.059060e+07
       lag_24h 5.201973e+06
pu_location_id 3.155564e+06
   pickup_hour 1.263878e+06
     trip_year 8.811183e+05
   day_of_week 4.653715e+05
      is_night 2.378662e+05
    is_weekend 1.342486e+05
         month 9.153847e+04
  is_rush_hour 2.848259e+04


### Save model evaluation metrics to feed model evaluation dashboard

In [17]:
# Scored predictions
(
    spark.createDataFrame(test_df[
        ["pu_location_id", "ds", "trip_count", "prediction",
         "is_rush_hour", "is_weekend", "pickup_hour"]
    ].rename(columns={"ds": "datetime"}))
    .write.format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")
    .save(r"..\forecasting_demand\model_evaluation_data\demand_predictions")
)

# Per-zone metrics
(
    spark.createDataFrame(per_zone_df)
    .write.format("delta")
    .mode("overwrite")
    .save(r"..\forecasting_demand\model_evaluation_data\forecast_metrics")
)

# Feature importances
(
    spark.createDataFrame(importance_df)
    .write.format("delta")
    .mode("overwrite")
    .save(r"..\forecasting_demand\model_evaluation_data\forecast_feature_importance")
)

logging.info("All platinum outputs written. Forecast complete.")

### Save model for future inference (e.g. batch scoring, real-time API)

In [18]:
import joblib

model_package = {
    "model": model,
    "features": FEATURE_COLS
}

BASE_DIR = os.path.abspath("") 

# Simplified path mapping based on the notebook's location
model_dir = os.path.join(BASE_DIR, "model")
os.makedirs(model_dir, exist_ok=True) # Ensure directory exists
model_path = os.path.join(model_dir, "demand_forecast_lgbm.pkl")

joblib.dump(model_package, model_path)

logging.info(f"Model saved to {model_path}")
print(f"\nModel saved to {model_path}")


Model saved to c:\Users\gauth\Desktop\Extended Medallion Architecture\products\forecasting_demand\model\demand_forecast_lgbm.pkl
